In [1]:
import os

In [2]:
%pwd


'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project'

In [6]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir, "kidney-ct-scan-image")
        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE
        )

        return training_config

In [9]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [10]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [11]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e

[2026-09-09 02:26:43,446: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-09 02:26:44,745: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-09 02:26:45,366: INFO: common: created directory at: artifacts]
[2026-09-09 02:26:45,372: INFO: common: created directory at: artifacts\training]
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
368/368 [==============================] - 2292s 6s/step - loss: 10.5646 - accuracy: 0.6201 - val_loss: 4.9301 - val_accuracy: 0.6937


c:\Users\PC\.conda\envs\kidney\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [13]:
import yaml

with open(CONFIG_FILE_PATH, "r", encoding="utf-8") as f:
    content = yaml.safe_load(f)

print(content)
print("\nTraining section:")
print(content["training"])

{'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://drive.google.com/file/d/1uuUV6qlbmSedb1GSqR0YfUwdZxFijNAH/view?usp=drive_link', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'prepare_base_model': {'root_dir': 'artifacts/prepare_base_model', 'base_model_path': 'artifacts/prepare_base_model/base_model.h5', 'updated_base_model_path': 'artifacts/prepare_base_model/base_model_updated.h5'}, 'training': {'root_dir': 'artifacts/training', 'trained_model_path': 'artifacts/training/model.h5'}}

Training section:
{'root_dir': 'artifacts/training', 'trained_model_path': 'artifacts/training/model.h5'}


In [14]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project'

In [9]:
import os

print(os.getcwd())

e:\KrishNair\Kidney_Disease_Classification_Deep_Learning_Project\research


In [15]:
config = ConfigurationManager()

training_config = config.get_training_config()

print(training_config)

[2026-09-09 02:09:32,215: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-09 02:09:32,229: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-09 02:09:32,234: INFO: common: created directory at: artifacts]
[2026-09-09 02:09:32,238: INFO: common: created directory at: artifacts\training]
TrainingConfig(root_dir=WindowsPath('artifacts/training'), trained_model_path=WindowsPath('artifacts/training/model.h5'), updated_base_model_path=WindowsPath('artifacts/prepare_base_model/base_model_updated.h5'), training_data=WindowsPath('artifacts/data_ingestion/kidney-ct-scan-image'), params_epochs=1, params_batch_size=16, params_is_augmentation=True, params_image_size=BoxList([224, 224, 3]))


In [19]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()

    training = Training(config=training_config)

    training.get_base_model()
    training.train_valid_generator()
    training.train()

except Exception as e:
    raise e

[2026-09-09 02:14:58,753: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-09 02:14:58,763: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-09 02:14:58,767: INFO: common: created directory at: artifacts]
[2026-09-09 02:14:58,769: INFO: common: created directory at: artifacts\training]
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.


NameError: name 'scipy' is not defined

In [17]:
%pip install Pillow

  Using cached pillow-10.4.0-cp38-cp38-win_amd64.whl.metadata (9.3 kB)
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.6 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.6 MB 1.3 MB/s eta 0:00:02
   ------------ --------------------------- 0.8/2.6 MB 1.4 MB/s eta 0:00:02
   ------------------------ --------------- 1.6/2.6 MB 1.9 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 2.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [18]:
from PIL import Image

print("Pillow is working!")

Pillow is working!


In [20]:
%pwd

'e:\\KrishNair\\Kidney_Disease_Classification_Deep_Learning_Project'

In [17]:
%pip install scipy


  Using cached scipy-1.10.1-cp38-cp38-win_amd64.whl.metadata (58 kB)
   ---------------------------------------- 0.0/42.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/42.2 MB ? eta -:--:--
    --------------------------------------- 1.0/42.2 MB 3.3 MB/s eta 0:00:13
   -- ------------------------------------- 2.4/42.2 MB 4.3 MB/s eta 0:00:10
   -- ------------------------------------- 2.6/42.2 MB 4.4 MB/s eta 0:00:09
   -- ------------------------------------- 2.9/42.2 MB 3.3 MB/s eta 0:00:12
   --- ------------------------------------ 3.4/42.2 MB 2.8 MB/s eta 0:00:14
   --- ------------------------------------ 4.2/42.2 MB 3.0 MB/s eta 0:00:13
   ---- ----------------------------------- 5.2/42.2 MB 3.3 MB/s eta 0:00:12
   ----- ---------------------------------- 6.3/42.2 MB 3.5 MB/s eta 0:00:11
   ------ --------------------------------- 7.3/42.2 MB 3.6 MB/s eta 0:00:10
   -------- ------------------------------- 8.7/42.2 MB 3.8 MB/s eta 0:00:09
   --------- -------

In [18]:
import scipy

print(scipy.__version__)

1.10.1
